# Create Dataset

In [ ]:

import itertools
import torch
import transformers
from datasets import load_dataset
from dataclasses import dataclass

local_model_path: str = "facebook/layerskip-llama2-7B"

# initialize model
tokenizer = transformers.AutoTokenizer.from_pretrained(local_model_path)
# model = transformers.AutoModelForCausalLM.from_pretrained(
#     local_model_path,
#     use_safetensors=True,
#     device_map="auto",
#     torch_dtype=torch.float16,
# )

DATASET = "abisee/cnn_dailymail"
VERSIONS = {"v1": "1.0.0", "v2": "2.0.0", "v3": "3.0.0"}
SPLITS = {"train": "train", "val": "validation", "test": "test"}

# 1. Load the dataset - create shards
# Each row in a shard contains input_ids, input_text, 
def load_records(version, split, args):
    """Stream only the selected split from the selected dataset configuration."""
    dataset = load_dataset(
        args.dataset, name=VERSIONS[version], split=SPLITS[split],
        streaming=True, # revision=args.dataset_revision,
    )
    records = iter(dataset)
    if args.max_documents is not None:
        records = itertools.islice(records, args.max_documents)
    return records


Loading checkpoint shards: 100%|██████████| 3/3 [00:10<00:00,  3.40s/it]


In [ ]:
if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.bos_token
        
inputs = tokenizer(
        text,
        return_tensors="pt",
        add_special_tokens=True,
        padding=True,
    ).to(model.get_input_embeddings().weight.device)

In [ ]:
@dataclass
class Argument:
    dataset: str
    max_documents: int

args = Argument(dataset=DATASET, max_documents=3)

records = []
for record in load_records("v1", "test", args):
    records.append(record)

In [1]:

import itertools
import torch
import transformers
from datasets import load_dataset
from dataclasses import dataclass

local_model_path: str = "facebook/layerskip-llama2-7B"

# initialize model
tokenizer = transformers.AutoTokenizer.from_pretrained(local_model_path)
model = transformers.AutoModelForCausalLM.from_pretrained(
    local_model_path,
    use_safetensors=True,
    device_map="auto",
    torch_dtype=torch.float16,
)


/home/markphan24/anaconda3/envs/layer_skip/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 3/3 [00:09<00:00,  3.06s/it]


In [7]:
import inspect

import torch
import transformers
import torch.nn.functional as F

@torch.inference_mode()
def forward_layer_by_layer(model, input_ids, attention_mask=None):
    model.eval()
    backbone = model.model

    # 1. Token IDs -> embeddings.
    # Shape: [batch_size, sequence_length, hidden_size]
    hidden_states = backbone.embed_tokens(input_ids)

    batch_size, seq_len, _ = hidden_states.shape
    device = hidden_states.device

    print("Embeddings:", tuple(hidden_states.shape))

    # 2. Token positions. This example does not use a KV cache.
    cache_position = torch.arange(seq_len, device=device)
    position_ids = cache_position.unsqueeze(0)

    # 3. Build the causal attention mask.
    # 0 = allowed; large negative value = blocked.
    mask_value = torch.finfo(hidden_states.dtype).min

    causal_mask = torch.full(
        (seq_len, seq_len),
        fill_value=mask_value,
        dtype=hidden_states.dtype,
        device=device,
    ).triu(diagonal=1)

    # Shape: [batch_size, 1, sequence_length, sequence_length]
    causal_mask = causal_mask[None, None].expand(
        batch_size, 1, seq_len, seq_len
    )

    # Also block attention to padding tokens.
    if attention_mask is not None:
        padding_mask = attention_mask.to(device)[:, None, None, :] == 0
        causal_mask = causal_mask.masked_fill(padding_mask, mask_value)

    # 4. Prepare rotary position embeddings.
    # Older Transformers versions compute these inside attention instead.
    position_embeddings = None
    if hasattr(backbone, "rotary_emb"):
        position_embeddings = backbone.rotary_emb(
            hidden_states, position_ids
        )

    # 5. Run decoder layers sequentially.
    layer_hidden_states = []
    layer_hidden_logits = []
    layer_hidden_outputs = []

    print(causal_mask.shape)
    print(causal_mask)

    for layer_number, layer in enumerate(backbone.layers, start=1):
        # Handle differences between Transformers layer APIs.
        parameters = inspect.signature(layer.forward).parameters

        kwargs = {
            "attention_mask": causal_mask,
            "position_ids": position_ids,
        }

        if "cache_position" in parameters:
            kwargs["cache_position"] = cache_position

        if "position_embeddings" in parameters:
            kwargs["position_embeddings"] = position_embeddings

        if "use_cache" in parameters:
            kwargs["use_cache"] = False

        if "output_attentions" in parameters:
            kwargs["output_attentions"] = False

        layer_output = layer(hidden_states, **kwargs)

        # Older versions return a tuple; newer versions return a tensor.
        hidden_states = (
            layer_output[0]
            if isinstance(layer_output, tuple)
            else layer_output
        )

        # Store raw decoder outputs on CPU.
        layer_hidden_states.append(
            hidden_states.detach().cpu().clone()
        )

        print(
            f"Layer {layer_number:02d}: "
            f"{tuple(hidden_states.shape)}"
        )

        # 6. Final normalization.
        normalized_hidden_states = backbone.norm(hidden_states)
    
        # 7. LM head: hidden vectors -> vocabulary logits.
        # Shape: [batch_size, sequence_length, vocab_size]
        logits = model.lm_head(normalized_hidden_states)
        layer_hidden_logits.append(logits)

        output_ids = F.softmax(logits, dim=-1)
        layer_hidden_outputs.append(output_ids)

    

    return layer_hidden_logits, layer_hidden_states


if __name__ == "__main__":
    model_path = "facebook/layerskip-llama2-7B"

    tokenizer = transformers.AutoTokenizer.from_pretrained(model_path)

    model = transformers.AutoModelForCausalLM.from_pretrained(
        model_path,
        use_safetensors=True,
        device_map="auto",
        torch_dtype=(
            torch.float16
            if torch.cuda.is_available()
            else torch.float32
        ),
        # Supports the explicit 4-D additive mask constructed above.
        # attn_implementation="eager",
    )

    text = ["The capital of France is", "Deep Learning is ", "Tom Selleck is an "]

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.bos_token

    inputs = tokenizer(
        text,
        return_tensors="pt",
        add_special_tokens=True,
        padding=True,
    ).to(model.get_input_embeddings().weight.device)

    print(inputs["input_ids"])
    print(inputs["attention_mask"])

    layer_hidden_logits, layer_hidden_states = forward_layer_by_layer(
        model,
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
    )
    # Assumes tokenizer.padding_side == "right".
    last_positions = inputs["attention_mask"].sum(dim=1) - 1

    for layer_number, logits in enumerate(layer_hidden_logits, start=1):
        batch_indices = torch.arange(logits.shape[0], device=logits.device)

        # Select each prompt's last non-padding token.
        last_logits = logits[
            batch_indices,
            last_positions.to(logits.device),
            :,
        ]

        probabilities = torch.softmax(last_logits.float(), dim=-1)

        # Both tensors have shape [3]: one result per prompt.
        next_token_probs, next_token_ids = probabilities.max(dim=-1)

        print(f"\nLayer {layer_number}")

        for i, prompt in enumerate(text):
            # Index one element BEFORE calling .item().
            token_id = next_token_ids[i].item()
            probability = next_token_probs[i].item()

            print(f"Prompt: {prompt!r}")
            print(f"Next token ID: {token_id}")
            print(f"Next token: {tokenizer.decode([token_id])!r}")
            print(f"Probability: {probability:.6f}")

    # Example: layer 8, first batch item, last prompt token.
    h8_last = layer_hidden_states[7][0, -1, :]
    print("Layer 8, last token:", tuple(h8_last.shape))  # (4096,)

/home/markphan24/anaconda3/envs/layer_skip/lib/python3.10/site-packages/accelerate/utils/modeling.py:1390: UserWarning: Current model requires 128 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.
  warnings.warn(
Loading checkpoint shards: 100%|██████████| 3/3 [00:00<00:00, 20.45it/s]


tensor([[    1,     1,     1,   450,  7483,   310,  3444,   338],
        [    1,     1,     1,     1, 21784, 29257,   338, 29871],
        [    1,  4335,   317,  1808,   384,   338,   385, 29871]])
tensor([[0, 0, 1, 1, 1, 1, 1, 1],
        [0, 0, 0, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1]])
Embeddings: (3, 8, 4096)
torch.Size([3, 1, 8, 8])
tensor([[[[-65504., -65504., -65504., -65504., -65504., -65504., -65504.,
           -65504.],
          [-65504., -65504., -65504., -65504., -65504., -65504., -65504.,
           -65504.],
          [-65504., -65504.,      0., -65504., -65504., -65504., -65504.,
           -65504.],
          [-65504., -65504.,      0.,      0., -65504., -65504., -65504.,
           -65504.],
          [-65504., -65504.,      0.,      0.,      0., -65504., -65504.,
           -65504.],
          [-65504., -65504.,      0.,      0.,      0.,      0., -65504.,
           -65504.],
          [-65504., -65504.,      0.,      0.,      0.,      0.,      0.,
    

In [5]:
causal_mask.shape

NameError: name 'causal_mask' is not defined

One record can store all token-level results for one article’s highlight

| Field | Shape | Meaning |
|---|---|---|
| `article_id` | Scalar | Identifies the article |
| `input_ids` | `[article_size + highlight_size + padding]` | Tokenizer of Format:<bos> [Prompt: ] <article> <eos> [Summary: ] <bos> <highligh> <eos>  |
| `hidden_states` | `[T, 32, 4096]` | All layer hidden vectors used to predict each highlight token |
| `token_distributions` | `[T, 32, vocab_size]` | Each layer’s next-token distribution |
| `early_exit_token_ids` | `[T, 32]` | Each layer’s top-choice prediction |
| `matches_final` | `[T, 32]` | Whether each layer’s top choice matches the final layer |

In [12]:
import torch
from torch import nn

class EarlyExitGRU(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True, bidirectional=False,
        )
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        states, _ = self.gru(x)
        return self.head(states)



In [ ]:
x = torch.rand((32, 4096))

model = EarlyExitGRU(4096, 64)

y = model(x[:-1, :])
import torch.nn.functional as F
y = F.sigmoid(y)
y

In [ ]:
import torch
from torch import nn

class EarlyExitGRU(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, batch_first=True)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, hs):
        """ 
        hs: hidden layer output - [h1, h2, ..., hL-1] - sequence of LLM layer hidden output - (batch, length, each is 1xC)
        """
        _, final_state = self.gru(hs)  # type of RNN depends on input - in this case, it is many-to-one RNN
        return self.head(final_state)

In [ ]:
from pathlib import Path
import torch

torch_dir = Path(torch.__file__).parent

print("Python GRU:", torch_dir / "nn" / "modules" / "rnn.py")
print("C++ headers:", torch_dir / "include")
print("Compiled libraries:", torch_dir / "lib")
print("C extension:", torch._C.__file__)

In [1]:
from pathlib import Path
import torch

torch_dir = Path(torch.__file__).parent

print("Compiled Python extension:", torch._C.__file__)
print("Backend libraries:", torch_dir / "lib")

for path in sorted((torch_dir / "lib").iterdir()):
    print(path)

Compiled Python extension: /home/markphan24/anaconda3/envs/layer_skip/lib/python3.10/site-packages/torch/_C.cpython-310-x86_64-linux-gnu.so
Backend libraries: /home/markphan24/anaconda3/envs/layer_skip/lib/python3.10/site-packages/torch/lib
/home/markphan24/anaconda3/envs/layer_skip/lib/python3.10/site-packages/torch/lib/libc10.so
/home/markphan24/anaconda3/envs/layer_skip/lib/python3.10/site-packages/torch/lib/libc10_cuda.so
/home/markphan24/anaconda3/envs/layer_skip/lib/python3.10/site-packages/torch/lib/libcaffe2_nvrtc.so
/home/markphan24/anaconda3/envs/layer_skip/lib/python3.10/site-packages/torch/lib/libgomp.so.1
/home/markphan24/anaconda3/envs/layer_skip/lib/python3.10/site-packages/torch/lib/libshm
/home/markphan24/anaconda3/envs/layer_skip/lib/python3.10/site-packages/torch/lib/libshm.so
/home/markphan24/anaconda3/envs/layer_skip/lib/python3.10/site-packages/torch/lib/libshm_windows
/home/markphan24/anaconda3/envs/layer_skip/lib/python3.10/site-packages/torch/lib/libtorch.so
/h